# Rolling Percentile-Rank Signal Sweep on BTC Hourly Bars

Adapted from the methodology in *Predicting Short-Term Crypto Returns with Market Microstructure* (aperiodic.io). The original notebook:

1. Pulls L2-imbalance microstructure metrics for BTC perpetuals (Binance futures, 5m bars).
2. Transforms each raw feature into a `[-1, +1]` signal via a rolling percentile rank: `signal_t = 2 * (rank_t - 1) / (window - 1) - 1`, where `rank_t` is computed inside `feature[t-w+1 : t+1]`.
3. Treats the signal as a position; the payoff is `ret_{t→t+1}`.
4. Sweeps every `(feature, window)` combination, ranks by Sharpe.
5. Reports the best combo's equity curve, IC and decile chart.

### Bias audit of the original

**The mechanical alignment is clean** — `rolling(w).rank()` is backward-looking (uses only points up to and including `t`), and the target is `close.pct_change().shift(-1)`, so position at `t` correctly earns the bar `t → t+1` return. No time-leak in feature construction.

**The headline result is in-sample-optimized**, however:

- `direction = 1 if fit_corr >= 0 else -1` is fit using the *entire* sample's IC, then applied to the *same* sample for the backtest. The sign of every signal is chosen with knowledge of the future.
- The best (feature, window) pair is selected from 96 candidates by sorting the *in-sample* Sharpe and taking the top row. The reported Sharpe ≈ 17 on ~9 k 5-min bars is the maximum of 96 noisy estimates — pure multiple-testing / data-snooping bias.
- The decile chart at the end is plotted using the in-sample-selected signal on the same data.

So no time-leak, but very strong selection bias. The framework is fine as a screening tool (which the author notes), but the reported metrics are not OOS.

### What this notebook does differently

1. **Data**: BTCUSDT spot hourly bars from `live_trading/cache/hourly/` (we do not have L2 imbalance data). The feature set is OHLCV-derived microstructure proxies — signed-volume bars, range-position of close, body/wick ratios, volume and return z-scores. These are weaker than true L2 metrics, but they're the honest mapping of the methodology to data we have.
2. **Train/test split**: ~70/30 by time. `(feature, window)` selection and direction sign happen on **train only**; reported Sharpe / IC / decile chart are **test-only**. This is the minimal fix to the in-sample selection bias above.
3. **Costs**: 2 bps per side on signal turnover (= sum |Δposition| × cost). The original uses 0 bps.

In [ ]:
from __future__ import annotations
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

REPO = Path.cwd().resolve()
while not (REPO / 'live_trading').exists() and REPO != REPO.parent:
    REPO = REPO.parent
assert (REPO / 'live_trading').exists(), 'Could not locate repo root'

SYMBOL        = 'BTCUSDT'
BARS_PATH     = REPO / 'live_trading' / 'cache' / 'hourly' / f'{SYMBOL}_hourly.parquet'
TRAIN_FRAC    = 0.70
RANK_WINDOWS  = [50, 100, 200, 500, 1000]
COST_BPS_SIDE = 1.0
BARS_PER_YEAR = 24 * 365      # hourly

raw = pd.read_parquet(BARS_PATH).sort_index()
raw.index.name = 'time'
print(f'{SYMBOL} hourly: {len(raw):,} bars, {raw.index[0]} → {raw.index[-1]}')
raw.tail(3)

## 1. Build microstructure-proxy features from OHLCV

All features at bar `t` use only OHLCV data from bar `t` and earlier. The `pct_change` target on the next line is `shift(-1)`-ed so position at `t` earns return `t → t+1`.

In [ ]:
def build_features(df: pd.DataFrame) -> pd.DataFrame:
    o, h, l, c, v = df['Open'], df['High'], df['Low'], df['Close'], df['Volume']
    rng   = (h - l).replace(0, np.nan)
    body  = (c - o)
    ret   = np.log(c / c.shift(1))

    f = pd.DataFrame(index=df.index)
    # Range-based pressure proxies (closer to L1/L2 imbalance in spirit)
    f['close_in_range']  = (c - l) / rng                # 0 = closed at low, 1 = closed at high
    f['body_to_range']   = body / rng                   # signed bar dominance
    f['upper_wick']      = (h - np.maximum(o, c)) / rng
    f['lower_wick']      = (np.minimum(o, c) - l) / rng
    # Signed-volume proxy: volume * sign(body)
    f['signed_vol']      = v * np.sign(body)
    # Volume z-scores at multiple horizons
    for w in (24, 168, 720):
        m = v.rolling(w).mean()
        s = v.rolling(w).std()
        f[f'vol_z_{w}']  = (v - m) / s
    # Return z-scores at multiple horizons
    for w in (24, 168, 720):
        m = ret.rolling(w).mean()
        s = ret.rolling(w).std()
        f[f'ret_z_{w}']  = (ret - m) / s
    # Range z-scores (intra-bar realized vol proxy)
    for w in (24, 168, 720):
        rng_norm = rng / c
        m = rng_norm.rolling(w).mean()
        s = rng_norm.rolling(w).std()
        f[f'rng_z_{w}']  = (rng_norm - m) / s
    return f.replace([np.inf, -np.inf], np.nan)

feats = build_features(raw)
panel = raw[['Close']].join(feats)
panel['fwd_ret'] = panel['Close'].pct_change().shift(-1)
panel = panel.dropna(subset=['fwd_ret'])
feature_cols = [c for c in feats.columns]

print(f'Panel: {len(panel):,} rows, {len(feature_cols)} features')
print('Features:', feature_cols)
panel.head(3)

## 2. Train/test split and rank transform

Split by time: first 70 % of bars are TRAIN (sweep / selection), last 30 % are TEST (held out). The rank transform is rolling and only looks backwards, so applying it to the full series is safe — but we still **score** each `(feature, window)` only on the train slice, then **re-score** the chosen combo on test.

In [ ]:
def rolling_rank_signal(x: pd.Series, window: int) -> pd.Series:
    """Rolling percentile rank in [-1, +1]. Backward-looking by construction."""
    rank = x.rolling(window).rank(method='average')
    return ((rank - 1.0) / (window - 1)) * 2.0 - 1.0

split_i = int(len(panel) * TRAIN_FRAC)
split_ts = panel.index[split_i]
print(f'Train: {panel.index[0]} → {panel.index[split_i-1]}  ({split_i:,} bars)')
print(f'Test : {panel.index[split_i]} → {panel.index[-1]}  ({len(panel)-split_i:,} bars)')

## 3. Sweep on TRAIN, then evaluate the winner on TEST

For each `(feature, window)` we compute the rank-signal, force `direction = sign(IC_train)`, and score net-Sharpe on train. We rank by train Sharpe, then apply the same direction sign to the test signal and report test-only metrics.

In [ ]:
def backtest(position: pd.Series, fwd_ret: pd.Series, cost_bps_side: float) -> dict:
    pos = position.fillna(0.0).clip(-1.0, 1.0)
    gross = pos * fwd_ret
    turnover = pos.diff().abs().fillna(pos.abs().iloc[0])
    cost = turnover * (cost_bps_side * 1e-4)
    net = gross - cost
    mu, sd = net.mean(), net.std()
    sharpe = (mu / sd) * np.sqrt(BARS_PER_YEAR) if sd > 0 else np.nan
    eq = (1.0 + net.fillna(0.0)).cumprod()
    mdd = float((eq / eq.cummax() - 1.0).min())
    return {
        'sharpe': float(sharpe),
        'mean_ret_bps': float(mu * 1e4),
        'total_return': float(eq.iloc[-1] - 1.0),
        'max_drawdown': mdd,
        'turnover_per_bar': float(turnover.mean()),
    }, eq, net

fwd_train = panel['fwd_ret'].iloc[:split_i]
fwd_test  = panel['fwd_ret'].iloc[split_i:]

rows = []
for feat in feature_cols:
    for w in RANK_WINDOWS:
        sig_full = rolling_rank_signal(panel[feat], w)
        sig_train, sig_test = sig_full.iloc[:split_i], sig_full.iloc[split_i:]
        mask_tr = sig_train.notna() & fwd_train.notna()
        if mask_tr.sum() < 200:
            continue
        ic_tr = float(np.corrcoef(sig_train[mask_tr], fwd_train[mask_tr])[0, 1])
        direction = 1 if ic_tr >= 0 else -1
        m_tr, _, _ = backtest(sig_train * direction, fwd_train, COST_BPS_SIDE)
        m_te, _, _ = backtest(sig_test  * direction, fwd_test,  COST_BPS_SIDE)
        mask_te = sig_test.notna() & fwd_test.notna()
        ic_te = float(np.corrcoef(sig_test[mask_te], fwd_test[mask_te])[0, 1]) if mask_te.sum() > 100 else np.nan
        rows.append({
            'feature': feat, 'window': w, 'direction': direction,
            'ic_train': ic_tr, 'sharpe_train': m_tr['sharpe'],
            'ic_test':  ic_te, 'sharpe_test':  m_te['sharpe'],
            'total_return_test': m_te['total_return'],
            'max_dd_test': m_te['max_drawdown'],
            'turnover_per_bar': m_te['turnover_per_bar'],
        })

results = pd.DataFrame(rows).sort_values('sharpe_train', ascending=False).reset_index(drop=True)
print(f'Swept {len(results)} (feature, window) combos.\nTop 10 by TRAIN Sharpe:')
results.head(10)

## 4. Train-vs-Test scatter — how badly does ranking on train survive OOS?

If features were truly predictive, train Sharpe should be a positive predictor of test Sharpe. A regression line near `y = 0` would mean the in-sample ranking is noise.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(results['sharpe_train'], results['sharpe_test'], alpha=0.7)
ax.axhline(0, color='black', linewidth=0.7, alpha=0.5)
ax.axvline(0, color='black', linewidth=0.7, alpha=0.5)
lo = min(results['sharpe_train'].min(), results['sharpe_test'].min())
hi = max(results['sharpe_train'].max(), results['sharpe_test'].max())
ax.plot([lo, hi], [lo, hi], 'k--', linewidth=0.7, alpha=0.4, label='y = x')
ax.set_xlabel('Sharpe (train, net of costs)')
ax.set_ylabel('Sharpe (test, net of costs)')
ax.set_title(f'{SYMBOL} hourly — train vs test Sharpe across {len(results)} combos')
ax.legend()
ax.grid(alpha=0.2)
rho = results[['sharpe_train', 'sharpe_test']].corr().iloc[0, 1]
ax.text(0.02, 0.98, f'corr(train, test) = {rho:.2f}', transform=ax.transAxes, va='top')
plt.show()

## 5. Equity curve of the train-selected winner on TEST

This is the analogue of the original notebook's equity-curve cell — but it's now genuinely OOS.

In [ ]:
best = results.iloc[0]
feat_best, w_best, dir_best = best['feature'], int(best['window']), int(best['direction'])
print(f'Winner on train: feature={feat_best}, window={w_best}, direction={dir_best:+d}')
print(f'  train: IC={best["ic_train"]:+.4f}  Sharpe={best["sharpe_train"]:+.2f}')
print(f'  test : IC={best["ic_test"]:+.4f}  Sharpe={best["sharpe_test"]:+.2f}  '
      f'TotRet={best["total_return_test"]:+.2%}  MaxDD={best["max_dd_test"]:+.2%}')

sig_full = rolling_rank_signal(panel[feat_best], w_best) * dir_best
_, eq_test, net_test = backtest(sig_full.iloc[split_i:], fwd_test, COST_BPS_SIDE)
_, eq_tr,   _        = backtest(sig_full.iloc[:split_i], fwd_train, COST_BPS_SIDE)

fig, axes = plt.subplots(2, 1, figsize=(13, 7), sharex=True)
axes[0].plot(sig_full.index, sig_full, linewidth=0.6, color='tab:red')
axes[0].axhline(0, color='black', linewidth=0.5, alpha=0.5)
axes[0].axvline(split_ts, color='tab:blue', linewidth=1.0, alpha=0.6, label='train/test split')
axes[0].set_title(f'Signal: {feat_best}  |  window={w_best}  |  direction={dir_best:+d}')
axes[0].legend(loc='upper left'); axes[0].grid(alpha=0.2)

axes[1].plot(eq_tr.index,   eq_tr,   linewidth=1.0, color='tab:gray',  label=f'train  (Sharpe {best["sharpe_train"]:+.2f})')
axes[1].plot(eq_test.index, eq_test, linewidth=1.2, color='tab:green', label=f'test   (Sharpe {best["sharpe_test"]:+.2f})')
axes[1].axvline(split_ts, color='tab:blue', linewidth=1.0, alpha=0.6)
axes[1].set_title(f'Net equity curve (costs={COST_BPS_SIDE:.0f} bps/side)')
axes[1].legend(); axes[1].grid(alpha=0.2)
plt.tight_layout(); plt.show()

## 6. Decile chart of signal vs forward return — TEST only

In [ ]:
sig_test_only = sig_full.iloc[split_i:]
mask = sig_test_only.notna() & fwd_test.notna()
sv, rv = sig_test_only[mask].to_numpy(), fwd_test[mask].to_numpy()
order  = np.argsort(sv)
deciles = np.empty(sv.shape[0], dtype=np.int64)
deciles[order] = (np.arange(sv.shape[0]) * 10 // sv.shape[0]) + 1

tbl = pd.DataFrame([
    {
        'decile': d,
        'count':  int((deciles == d).sum()),
        'mean_signal':  float(np.nanmean(sv[deciles == d])),
        'mean_fwd_ret_bps': float(np.nanmean(rv[deciles == d]) * 1e4),
    }
    for d in range(1, 11)
])

fig, ax = plt.subplots(figsize=(10, 4))
colors = ['tab:red' if v < 0 else 'tab:blue' for v in tbl['mean_fwd_ret_bps']]
ax.bar(tbl['decile'], tbl['mean_fwd_ret_bps'], color=colors, alpha=0.85)
ax.axhline(0, color='black', linewidth=0.6, alpha=0.6)
ax.set_title(f'TEST: mean next-bar return (bps) by signal decile — {feat_best}, w={w_best}')
ax.set_xlabel('Signal decile (1 = most negative, 10 = most positive)')
ax.set_ylabel('Mean fwd return (bps)')
ax.grid(alpha=0.2, axis='y')
plt.tight_layout(); plt.show()
tbl

## 7. Notes

- The rank-transform is the genuinely portable contribution of the original — it turns any time series into a stationary, scale-free, near-uniform position. It costs almost nothing to add to an existing feature stack.
- What does **not** transfer is the headline Sharpe. Picking the best of N candidates on the same data you score on is a free Sharpe inflator. The train/test split here is the cheapest fix; a walk-forward selection (see [3_walkforward.ipynb](3_walkforward.ipynb)) is the more honest one.
- OHLCV-derived features are a much weaker substitute for true L2-imbalance metrics. If we wire up Aperiodic or another L2 feed, the same notebook with the same selection protocol should produce a stronger out-of-sample edge.